# Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario is an exact replay of history: `simulated_flows_df == historical_flows_df`.

In [1]:
import pandas as pd

from dataloader_raw import RawModelData
from dataloader_graph import ResolvedModelData, attach_simulation
from engine import Environment, EnvironmentConfig
from phases import (
    ArrivalsPreviousPhase,
    OverflowRedirectPreviousPhase,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
    ArrivalsPhase,
    OverflowRedirectPhase,
)

In [2]:
# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path="../data/raw/202602-citibike-tripdata_1.csv",
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

# Graph (resolved) model data: period grid, historical flows, replay demand.
graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1))

historical_flows_df = graph_data.historical_flows_df

In [ ]:
phases_canonical = [
    ArrivalsPreviousPhase(),
    OverflowRedirectPreviousPhase(),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    ArrivalsPhase(),
    OverflowRedirectPhase(),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, seed=42, scenario_id="historical_replay"),
)
env_canonical.run()

# Wire the finished run back into the graph-data container's simulated_* slots.
attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_inventory_df = graph_data.simulated_inventory_df

# The whole point of the base scenario: an exact replay of history.
pd.testing.assert_frame_equal(simulated_flows_df, historical_flows_df)
print("simulated_flows_df == historical_flows_df:",
      simulated_flows_df.equals(historical_flows_df))

simulated_flows_df == historical_flows_df: True


In [5]:
print("simulated_flows_df == historical_flows_df:",
      simulated_flows_df.equals(historical_flows_df))

simulated_flows_df == historical_flows_df: True


In [4]:
pd.testing.assert_frame_equal(simulated_flows_df, historical_flows_df)


In [4]:
simulated_flows_df

,event_id,period_id,flow_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason
0,0,0,hist_244666,user_trip,departed,classic_bike,5210.01,5411.08,<NA>,0,23,<NA>,<NA>,1,<NA>
1,1,3,hist_198173,user_trip,departed,classic_bike,6575.03,6659.01,<NA>,3,23,<NA>,<NA>,1,<NA>
2,2,3,hist_277917,user_trip,departed,classic_bike,7484.05,6659.01,<NA>,3,23,<NA>,<NA>,1,<NA>
3,3,6,hist_177716,user_trip,departed,classic_bike,4550.05,4830.02,<NA>,6,25,<NA>,<NA>,1,<NA>
4,4,7,hist_253909,user_trip,departed,classic_bike,4196.05,4425.02,<NA>,7,22,<NA>,<NA>,1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1988285,1988285,686,hist_993009,user_trip,departed,electric_bike,5238.05,5288.09,<NA>,686,686,<NA>,<NA>,1,<NA>
1988286,1988286,686,hist_993009,user_trip,arrived,electric_bike,5238.05,5288.09,5288.09,686,686,686,<NA>,1,<NA>
1988287,1988287,686,hist_993623,user_trip,departed,electric_bike,4386.05,4611.03,<NA>,686,686,<NA>,<NA>,1,<NA>
1988288,1988288,686,hist_993623,user_trip,arrived,electric_bike,4386.05,4611.03,4611.03,686,686,686,<NA>,1,<NA>
